# Broker pesan — AMQP 1.0 dalam proses, NATS dan Kafka lewat uraian

NATS dan Kafka butuh broker sungguhan, jadi notebook ini menjalankan adapter AMQP 1.0 terhadap `AmqpMiniBroker`, broker kecil dalam proses untuk pengujian dan demo (bukan untuk produksi). Adapter NATS dan Kafka mengikuti model endpoint yang sama; cara pakainya ditunjukkan dalam teks di bawah.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Amqp, 0.22.0-preview.1"

## Broker dan gateway
Mini broker menyimpan satu antrean dalam memori per address dan memeriksa kredensial SASL PLAIN bila Anda memberikannya. Kredensial adalah rahasia: yang di sini hanya nilai demo.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Amqp;
using System.Text;

var broker = AmqpMiniBroker.Create(userName: "plant", password: "demo-only");
await broker.StartAsync();
var gateway = AmqpEndpoint.Create(o => o.UseBroker(broker.Address).WithCredentials("plant", "demo-only"));
await gateway.ConnectAsync();
Console.WriteLine($"connected: {gateway.IsConnected}");

for (var i = 0; i < 3; i++)
    await gateway.PublishAsync("plant.temp", Encoding.UTF8.GetBytes($"2{i}.5"), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce, ContentType = "text/plain" });
Console.WriteLine($"waiting on plant.temp: {broker.QueueLength("plant.temp")}");

## Subscribe
Subscription adalah receiver link. Setiap pesan diterima (accepted) ketika loop meminta pesan berikutnya.

In [ ]:
var cts = new CancellationTokenSource(TimeSpan.FromSeconds(10));
var got = new List<string>();
await foreach (var m in gateway.SubscribeAsync("plant.temp", cts.Token))
{
    got.Add(Encoding.UTF8.GetString(m.Payload.Span));
    if (got.Count == 3) break;
}
Console.WriteLine(string.Join(", ", got));

## Pengiriman ulang dan penolakan
Pesan yang belum selesai diproses konsumen kembali ke antrean. Broker yang menolak pesan membuat publish terkonfirmasi melempar exception; publish pre-settled tidak menunggu hasilnya.

In [ ]:
await gateway.PublishAsync("plant.alarm", "overheat"u8.ToArray(), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce });
await foreach (var m in gateway.SubscribeAsync("plant.alarm", cts.Token))
{
    Console.WriteLine($"took {Encoding.UTF8.GetString(m.Payload.Span)} and stopped without finishing");
    break;
}
await foreach (var m in gateway.SubscribeAsync("plant.alarm", cts.Token))
{
    Console.WriteLine($"delivered again: {Encoding.UTF8.GetString(m.Payload.Span)}");
    break;
}

broker.RejectedAddresses.Add("plant.closed");
try { await gateway.PublishAsync("plant.closed", "x"u8.ToArray(), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce }); }
catch (DeviceException e) { Console.WriteLine("confirmed publish failed: " + e.Message); }
await gateway.PublishAsync("plant.closed", "x"u8.ToArray());   // pre-settled: no outcome awaited
Console.WriteLine($"messages accepted by consumers so far: {broker.MessagesDelivered}");

## Kredensial salah
Broker memeriksanya, dan kegagalannya berupa `TransportException`.

In [ ]:
var intruder = AmqpEndpoint.Create(o => o.UseBroker(broker.Address).WithCredentials("plant", "wrong"));
try { await intruder.ConnectAsync(); } catch (TransportException e) { Console.WriteLine("refused: " + e.GetType().Name); }
await intruder.DisposeAsync();
await gateway.DisposeAsync();
await broker.DisposeAsync();

## NATS dan Kafka
Keduanya butuh broker sungguhan, jadi hanya ditampilkan sebagai teks. Keduanya adalah endpoint seperti di atas (`ConnectAsync`, `PublishAsync`, `SubscribeAsync`, traffic tap, helper hosting `AddNats` dan `AddKafka`).

```csharp
// NATS: wildcard, queue group, dan request/reply
using IoTCom.Net.Adapters.Nats;

await using var nats = NatsEndpoint.Create(o => o.UseServer("nats://localhost:4222"));
await nats.ConnectAsync();
await nats.PublishAsync("plant.line1.temp", "21.5"u8.ToArray());
await foreach (var m in nats.ReceiveAsync("plant.>", queueGroup: "loggers")) { /* setiap pesan sampai ke satu anggota grup */ }
var reply = await nats.RequestAsync("plant.line1.info", ReadOnlyMemory<byte>.Empty);
```

```csharp
// Kafka: key, header, consumer group, topic ekspresi reguler (filter yang diawali ^)
using IoTCom.Net.Adapters.Kafka;

await using var kafka = KafkaEndpoint.Create(o => o.UseBootstrap("localhost:9092").WithGroup("gateway"));
await kafka.ConnectAsync();
await kafka.PublishAsync("plant-temp", key: "line1"u8.ToArray(), value: "21.5"u8.ToArray());
await foreach (var r in kafka.ReceiveAsync("^plant-.*")) Console.WriteLine($"{r.Topic}[{r.Partition}]@{r.Offset}");
```

Pengujian integrasinya berjalan bila `IOTCOM_NATS_URL` dan `IOTCOM_KAFKA_BOOTSTRAP` disetel.

## Lebih lanjut
Lihat `docs/id/protocols/messaging-brokers.md` dan `docs/id/protocols/zenoh.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*